# Notebook 06b: Product Recommendation Systems

**Business objective:** Suggest products to each customer to increase basket size and repeat purchase rate.

I build and compare three recommendation architectures, each using a different technique to model the relationship between customers and product categories:

1. **SVD (Matrix Factorisation)** — collaborative filtering baseline; decomposes the user-item rating matrix into latent factor vectors
2. **Neural Collaborative Filtering (NCF)** — learns user and item embeddings through a deep MLP; captures non-linear preference patterns that factorisation misses
3. **Wide and Deep** — extends NCF with explicit side features (engagement score, category exploration, brand diversity, transaction count); the wide path memorises direct feature effects while the deep path learns interactions

I use the transaction-level enriched dataset from notebook 3, which includes pre-computed engagement and diversity scores. These features give the models behavioural context beyond raw ratings.

## Step 1: Imports and Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import pickle
import joblib
from datetime import datetime
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Fix random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Timestamp-based version string to uniquely label all saved artefacts
MODEL_VERSION = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f"Model version: {MODEL_VERSION}")

Model version: 20260227_171329


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, ndcg_score

# Surprise provides the SVD implementation and cross-validation helpers
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate, GridSearchCV

# TensorFlow for NCF and Wide & Deep
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow: {tf.__version__}")

TensorFlow: 2.13.0


## Step 2: Load Transaction-Level Enriched Data

I use `transaction_level_enriched.csv` rather than the raw transaction file because it already has all the per-customer behavioural scores (engagement, category exploration, brand diversity, repeat buyer) merged in. Using pre-computed scores avoids recalculating them here and keeps this notebook focused on modelling.

In [3]:
# Load the enriched transaction dataset produced by notebook 3
df = pd.read_csv('../data/transaction_level_enriched.csv')
print(f"Dataset shape        : {df.shape}")
print(f"Total transactions   : {len(df):,}")
print(f"Unique customers     : {df['Customer_ID'].nunique():,}")

# Quick check that the key enriched features are present
enriched_cols = ['Customer_Engagement_Score', 'Category_Exploration_Score',
                 'Brand_Diversity_Score', 'Repeat_Buyer_Score']
print(f"\nEnriched feature summary:")
print(df[enriched_cols].describe())

Dataset shape        : (301006, 115)
Total transactions   : 301,006
Unique customers     : 86,740

Enriched feature summary:
       Customer_Engagement_Score  Category_Exploration_Score  \
count              301006.000000               301006.000000   
mean                   10.459084                    0.953540   
std                     3.120900                    0.378514   
min                     3.000000                    0.000000   
25%                     8.000000                    0.693147   
50%                    11.000000                    1.039721   
75%                    13.000000                    1.273028   
max                    15.000000                    1.609438   

       Brand_Diversity_Score  Repeat_Buyer_Score  
count          301006.000000       301006.000000  
mean                3.818047            1.617412  
std                 1.513565            0.360914  
min                 1.000000            0.693147  
25%                 3.000000            1.3

In [4]:
# Select the columns needed to build user-item interactions
# I drop rows without a rating because Surprise cannot impute them
interaction_cols = [
    'Customer_ID', 'Product_Category', 'Ratings', 'Amount',
    'Customer_Engagement_Score', 'Category_Exploration_Score',
    'Brand_Diversity_Score', 'Repeat_Buyer_Score',
    'Avg_Rating', 'Transaction_Count',
]
interaction_df = df[interaction_cols].copy()
interaction_df = interaction_df.dropna(subset=['Ratings'])

print(f"Interaction rows     : {len(interaction_df):,}")
print(f"Unique customers     : {interaction_df['Customer_ID'].nunique():,}")
print(f"Unique categories    : {interaction_df['Product_Category'].nunique()}")
print(f"\nRating distribution:")
print(interaction_df['Ratings'].value_counts().sort_index())

Interaction rows     : 301,006
Unique customers     : 86,740
Unique categories    : 5

Rating distribution:
Ratings
1.0    43155
1.4        1
1.6        3
1.8        3
2.0    62480
2.2        3
2.4       10
2.6       17
2.8       15
3.0    47523
3.2       31
3.4       21
3.6       17
3.8       12
4.0    97742
4.2        4
4.4        3
4.8        1
5.0    49965
Name: count, dtype: int64


In [5]:
# Weight each rating by the customer's engagement score before averaging
# Customers who engage more with the platform have more reliable preference signals
interaction_df['weighted_rating'] = (
    interaction_df['Ratings'] * (1 + interaction_df['Customer_Engagement_Score'])
)

agg_dict = {
    'Ratings':                  'mean',
    'weighted_rating':          'sum',
    'Customer_Engagement_Score': 'mean',
    'Category_Exploration_Score': 'first',
    'Brand_Diversity_Score':    'first',
    'Transaction_Count':        'first',
    'Amount':                   'sum',
}

# Aggregate to one row per (customer, category) pair
user_item_ratings = (
    interaction_df
    .groupby(['Customer_ID', 'Product_Category'])
    .agg(agg_dict)
    .reset_index()
)

# Compute the final engagement-weighted rating
user_item_ratings['engagement_weight'] = 1 + user_item_ratings['Customer_Engagement_Score']
user_item_ratings['final_rating'] = (
    user_item_ratings['weighted_rating'] / user_item_ratings['engagement_weight']
)

print(f"User-item pairs      : {len(user_item_ratings):,}")
print(f"\nRating statistics after engagement weighting:")
print(user_item_ratings[['Ratings', 'final_rating', 'Customer_Engagement_Score']].describe())

User-item pairs      : 217,644

Rating statistics after engagement weighting:
             Ratings   final_rating  Customer_Engagement_Score
count  217644.000000  217644.000000              217644.000000
mean        3.162815       4.373737                  10.014538
std         1.211005       2.597531                   3.189939
min         1.000000       1.000000                   3.000000
25%         2.000000       2.000000                   8.000000
50%         3.000000       4.000000                  10.000000
75%         4.000000       5.000000                  13.000000
max         5.000000      26.000000                  15.000000


In [6]:
# Encode customer IDs and category names as consecutive integers
# Neural models require integer indices for their Embedding layers
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

user_item_ratings['user_id'] = user_encoder.fit_transform(user_item_ratings['Customer_ID'])
user_item_ratings['item_id'] = item_encoder.fit_transform(user_item_ratings['Product_Category'])

n_users = user_item_ratings['user_id'].nunique()
n_items = user_item_ratings['item_id'].nunique()

# Sparsity tells me how many possible (user, item) pairs have no observed rating
sparsity = 1 - len(user_item_ratings) / (n_users * n_items)
print(f"Encoded users        : {n_users:,}")
print(f"Encoded items        : {n_items}")
print(f"Interaction sparsity : {sparsity:.4f}  ({sparsity * 100:.2f}% of pairs unobserved)")

Encoded users        : 86,740
Encoded items        : 5
Interaction sparsity : 0.4982  (49.82% of pairs unobserved)


## Step 3: Stratified Train-Test Split

I split at the user level so that every user who appears in the test set also has interactions in the training set. This is the cold-start prevention principle: a model cannot predict for a user it has never seen. A simple random split does not guarantee this, which is why I use a custom stratified function.

In [7]:
def stratified_user_split(df, user_col='user_id', test_size=0.2, random_state=RANDOM_STATE):
    """Split interactions ensuring every test user has training history.

    For each user, 20% of their interactions go to the test set and the rest
    to training. Users with only one interaction are placed entirely in training
    to avoid cold-start users appearing in the test set.
    """
    rng = np.random.default_rng(random_state)
    train_indices, test_indices = [], []

    for _, group in df.groupby(user_col):
        idx = group.index.to_numpy()
        if len(idx) == 1:
            # Only one interaction — cannot split, keep in training
            train_indices.extend(idx.tolist())
            continue
        n_test = max(1, int(len(idx) * test_size))
        sampled_test = rng.choice(idx, size=n_test, replace=False)
        test_indices.extend(sampled_test.tolist())
        train_indices.extend([i for i in idx if i not in sampled_test])

    return (df.loc[train_indices].reset_index(drop=True),
            df.loc[test_indices].reset_index(drop=True))


train_data, test_data = stratified_user_split(user_item_ratings, test_size=0.2)

# Hard check — any cold-start user in test would invalidate predictions
cold_start = set(test_data['user_id']) - set(train_data['user_id'])
assert not cold_start, f"Cold-start users detected: {cold_start}"

print(f"Train interactions   : {len(train_data):,}")
print(f"Test interactions    : {len(test_data):,}")
print(f"Train unique users   : {train_data['user_id'].nunique():,}")
print(f"Test unique users    : {test_data['user_id'].nunique():,}")
print(f"Cold-start check     : passed (no unseen users in test)")

Train interactions   : 146,281
Test interactions    : 71,363
Train unique users   : 86,740
Test unique users    : 71,363
Cold-start check     : passed (no unseen users in test)


## Step 4: Baseline Performance

Before training any model I compute two naive baselines: (1) always predict the global mean rating, and (2) always predict each user's own mean historical rating. Any model I train must outperform the per-user mean baseline on both RMSE and MAE to be useful.

In [8]:
# Baseline 1: always predict the overall mean rating from the training set
global_mean_rating = train_data['final_rating'].mean()
global_baseline    = np.full(len(test_data), global_mean_rating)

# Baseline 2: predict each user's personal mean rating; fallback to global for unseen users
per_user_mean          = train_data.groupby('user_id')['final_rating'].mean()
personalized_baseline  = test_data['user_id'].map(per_user_mean).fillna(global_mean_rating)

rmse_global = np.sqrt(mean_squared_error(test_data['final_rating'], global_baseline))
mae_global  = mean_absolute_error(test_data['final_rating'],  global_baseline)
rmse_user   = np.sqrt(mean_squared_error(test_data['final_rating'], personalized_baseline))
mae_user    = mean_absolute_error(test_data['final_rating'],  personalized_baseline)

print("Baseline metrics — models must improve on these:")
print(f"  Global mean predictor  RMSE={rmse_global:.4f}  MAE={mae_global:.4f}")
print(f"  Per-user mean predictor RMSE={rmse_user:.4f}  MAE={mae_user:.4f}")

Baseline metrics — models must improve on these:
  Global mean predictor  RMSE=2.5939  MAE=1.9211
  Per-user mean predictor RMSE=3.3631  MAE=2.5151


## Step 5: Model 1 — SVD Matrix Factorisation

SVD decomposes the user-item rating matrix into lower-dimensional user and item latent factor matrices. Each user and item is represented as a dense vector in a shared latent space; the predicted rating is the dot product of their vectors plus user/item bias terms. I use the Surprise library's `GridSearchCV` to find the best combination of embedding dimension (`n_factors`), learning rate and regularisation.

In [9]:
# Surprise requires a Reader that declares the rating scale
reader       = Reader(rating_scale=(train_data['final_rating'].min(),
                                    train_data['final_rating'].max()))
surprise_data = Dataset.load_from_df(train_data[['user_id', 'item_id', 'final_rating']], reader)

# Grid search over a small hyperparameter space using 3-fold cross-validation
svd_param_grid = {
    'n_factors': [50, 100],      # latent embedding dimension
    'n_epochs':  [30, 50],
    'lr_all':    [0.003, 0.005],
    'reg_all':   [0.01, 0.02],   # regularisation applied to all parameters
}

print("Running SVD grid search (3-fold CV)...")
svd_grid = GridSearchCV(SVD, svd_param_grid, measures=['rmse'], cv=3, n_jobs=-1)
svd_grid.fit(surprise_data)
svd_best_params = svd_grid.best_params['rmse']
print(f"Best params: {svd_best_params}")

# Train the final model on the entire training set with the best params
trainset  = surprise_data.build_full_trainset()
svd_model = SVD(random_state=RANDOM_STATE, **svd_best_params)
svd_model.fit(trainset)
print("SVD training complete.")

Running SVD grid search (3-fold CV)...
Best params: {'n_factors': 100, 'n_epochs': 50, 'lr_all': 0.005, 'reg_all': 0.02}
SVD training complete.


In [10]:
# Generate predictions for each (user, item) pair in the test set
y_true_svd, y_pred_svd = [], []
for _, row in test_data.iterrows():
    pred = svd_model.predict(row['user_id'], row['item_id'])
    y_true_svd.append(row['final_rating'])
    y_pred_svd.append(pred.est)

rmse_svd = np.sqrt(mean_squared_error(y_true_svd, y_pred_svd))
mae_svd  = mean_absolute_error(y_true_svd,  y_pred_svd)

# Express improvement relative to the per-user mean baseline
rmse_improvement_svd = (rmse_user - rmse_svd) / rmse_user * 100
mae_improvement_svd  = (mae_user  - mae_svd)  / mae_user  * 100

print(f"SVD — RMSE={rmse_svd:.4f} ({rmse_improvement_svd:+.2f}% vs baseline)  "
      f"MAE={mae_svd:.4f} ({mae_improvement_svd:+.2f}% vs baseline)")

svd_metrics = {
    'rmse': float(rmse_svd),
    'mae':  float(mae_svd),
    'rmse_improvement_pct': float(rmse_improvement_svd),
    'mae_improvement_pct':  float(mae_improvement_svd),
}

SVD — RMSE=2.6185 (+22.14% vs baseline)  MAE=1.9506 (+22.45% vs baseline)


In [11]:
# Save SVD model and its metadata
svd_model_path = '../artifacts/models/recommendation_svd.pkl'
with open(svd_model_path, 'wb') as f:
    pickle.dump(svd_model, f)

svd_metadata = {
    'model_version':         MODEL_VERSION,
    'model_type':            'SVD Matrix Factorisation',
    'created_at':            datetime.now().isoformat(),
    'n_users':               int(n_users),
    'n_items':               int(n_items),
    'train_interactions':    int(len(train_data)),
    'test_interactions':     int(len(test_data)),
    'best_params':           svd_best_params,
    'metrics':               svd_metrics,
    'baseline_metrics':      {'global_rmse': float(rmse_global), 'user_rmse': float(rmse_user)},
}
with open('../artifacts/models/recommendation_svd_metadata.json', 'w') as f:
    json.dump(svd_metadata, f, indent=2)

print(f"SVD model saved   : {svd_model_path}")
print(f"SVD metadata saved: ../artifacts/models/recommendation_svd_metadata.json")

SVD model saved   : ../artifacts/models/recommendation_svd.pkl
SVD metadata saved: ../artifacts/models/recommendation_svd_metadata.json


## Step 6: Model 2 — Neural Collaborative Filtering (NCF)

NCF replaces the fixed dot-product interaction of SVD with a learned MLP. The user and item are each mapped to a dense embedding vector; those vectors are concatenated and passed through a stack of dense layers with batch normalisation and dropout. This lets the model capture asymmetric and non-linear preference interactions that a linear factorisation misses.

In [12]:
# Extract the integer-encoded user/item columns and the target ratings
X_train_ncf = train_data[['user_id', 'item_id']].values
y_train_ncf = train_data['final_rating'].values
X_test_ncf  = test_data[['user_id',  'item_id']].values
y_test_ncf  = test_data['final_rating'].values

print(f"NCF train shape: {X_train_ncf.shape}   test shape: {X_test_ncf.shape}")

NCF train shape: (146281, 2)   test shape: (71363, 2)


In [13]:
def build_ncf_model(n_users, n_items, embedding_size=64,
                    hidden_units=(256, 128, 64), dropout_rates=(0.3, 0.25, 0.2)):
    """NCF model: user and item embeddings → MLP → single rating output.

    he_normal initialisation and L2 regularisation on the embeddings help
    prevent the embedding vectors from growing too large during training.
    """
    user_input = Input(shape=(1,), name='user_input')
    # Embedding layer maps integer user IDs to dense vectors of length embedding_size
    user_vec = Flatten()(Embedding(n_users, embedding_size, name='user_embedding',
                                   embeddings_initializer='he_normal',
                                   embeddings_regularizer=regularizers.l2(1e-6))(user_input))

    item_input = Input(shape=(1,), name='item_input')
    item_vec = Flatten()(Embedding(n_items, embedding_size, name='item_embedding',
                                   embeddings_initializer='he_normal',
                                   embeddings_regularizer=regularizers.l2(1e-6))(item_input))

    # Concatenate both embeddings before the MLP layers
    x = Concatenate()([user_vec, item_vec])
    for units, dr in zip(hidden_units, dropout_rates):
        x = Dense(units, activation='relu', kernel_regularizer=regularizers.l2(1e-6))(x)
        x = BatchNormalization()(x)
        x = Dropout(dr)(x)

    # Linear output — regression over the continuous rating range
    output = Dense(1, activation='linear', name='output')(x)

    return Model(inputs=[user_input, item_input], outputs=output)


ncf_model = build_ncf_model(n_users, n_items, embedding_size=64)
ncf_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
ncf_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 user_input (InputLayer)     [(None, 1)]                  0         []                            
                                                                                                  
 item_input (InputLayer)     [(None, 1)]                  0         []                            
                                                                                                  
 user_embedding (Embedding)  (None, 1, 64)                5551360   ['user_input[0][0]']          
                                                                                                  
 item_embedding (Embedding)  (None, 1, 64)                320       ['item_input[0][0]']          
                                                                                              

In [14]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

print("Training NCF model...")
ncf_history = ncf_model.fit(
    [X_train_ncf[:, 0], X_train_ncf[:, 1]], y_train_ncf,
    validation_data=([X_test_ncf[:, 0], X_test_ncf[:, 1]], y_test_ncf),
    epochs=50,
    batch_size=512,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)
print(f"NCF training complete — {len(ncf_history.history['loss'])} epochs.")

Training NCF model...
Epoch 1/50
286/286 [==============================] - 22s 69ms/step - loss: 14.4967 - mae: 2.8892 - val_loss: 6.6885 - val_mae: 1.9239 - lr: 0.0010
Epoch 2/50
286/286 [==============================] - 18s 64ms/step - loss: 7.4679 - mae: 2.0429 - val_loss: 7.1818 - val_mae: 2.1033 - lr: 0.0010
Epoch 3/50
286/286 [==============================] - 19s 68ms/step - loss: 4.6291 - mae: 1.6026 - val_loss: 7.3817 - val_mae: 2.0710 - lr: 0.0010
Epoch 4/50
286/286 [==============================] - 19s 66ms/step - loss: 2.7713 - mae: 1.2493 - val_loss: 7.0318 - val_mae: 1.9125 - lr: 0.0010
Epoch 5/50
286/286 [==============================] - 19s 65ms/step - loss: 1.7600 - mae: 0.9867 - val_loss: 6.9718 - val_mae: 1.9643 - lr: 5.0000e-04
Epoch 6/50
286/286 [==============================] - 19s 68ms/step - loss: 1.1822 - mae: 0.8051 - val_loss: 7.0132 - val_mae: 1.9498 - lr: 5.0000e-04
NCF training complete — 6 epochs.


In [15]:
# Generate predictions on the test set
y_pred_ncf = ncf_model.predict([X_test_ncf[:, 0], X_test_ncf[:, 1]], verbose=0).flatten()

rmse_ncf = np.sqrt(mean_squared_error(y_test_ncf, y_pred_ncf))
mae_ncf  = mean_absolute_error(y_test_ncf, y_pred_ncf)

rmse_improvement_ncf = (rmse_user - rmse_ncf) / rmse_user * 100
mae_improvement_ncf  = (mae_user  - mae_ncf)  / mae_user  * 100

print(f"NCF — RMSE={rmse_ncf:.4f} ({rmse_improvement_ncf:+.2f}% vs baseline)  "
      f"MAE={mae_ncf:.4f} ({mae_improvement_ncf:+.2f}% vs baseline)")

ncf_metrics = {
    'rmse': float(rmse_ncf),
    'mae':  float(mae_ncf),
    'rmse_improvement_pct': float(rmse_improvement_ncf),
    'mae_improvement_pct':  float(mae_improvement_ncf),
}

NCF — RMSE=2.5859 (+23.11% vs baseline)  MAE=1.9239 (+23.51% vs baseline)


In [16]:
# Save NCF model as .h5 — the FastAPI backend expects this format
ncf_model_path = '../artifacts/models/recommendation_ncf.h5'
ncf_model.save(ncf_model_path)

ncf_metadata = {
    'model_version':      MODEL_VERSION,
    'model_type':         'Neural Collaborative Filtering',
    'created_at':         datetime.now().isoformat(),
    'n_users':            int(n_users),
    'n_items':            int(n_items),
    'embedding_size':     64,
    'train_interactions': int(len(train_data)),
    'metrics':            ncf_metrics,
    'training_epochs':    len(ncf_history.history['loss']),
}
with open('../artifacts/models/recommendation_ncf_metadata.json', 'w') as f:
    json.dump(ncf_metadata, f, indent=2)

print(f"NCF model saved   : {ncf_model_path}")
print(f"NCF metadata saved: ../artifacts/models/recommendation_ncf_metadata.json")

NCF model saved   : ../artifacts/models/recommendation_ncf.h5
NCF metadata saved: ../artifacts/models/recommendation_ncf_metadata.json


## Step 7: Model 3 — Wide and Deep Recommender

This model adds the four pre-computed behavioural features as explicit inputs alongside the user and item embeddings. The wide part is a small dense layer applied directly to these features (linear memorisation), while the deep part continues from the embedded representations. Combining them lets the model use structured business signals alongside the latent collaborative signal.

In [17]:
from sklearn.preprocessing import StandardScaler

# The four side features that go into the wide input path
feature_cols = ['Customer_Engagement_Score', 'Category_Exploration_Score',
                'Brand_Diversity_Score', 'Transaction_Count']

X_train_features = train_data[feature_cols].fillna(0).values
X_test_features  = test_data[feature_cols].fillna(0).values

# Scale features using training statistics only — no test-set leakage
feature_scaler = StandardScaler()
X_train_features_scaled = feature_scaler.fit_transform(X_train_features)
X_test_features_scaled  = feature_scaler.transform(X_test_features)

print(f"Side feature shape : {X_train_features_scaled.shape}")

Side feature shape : (146281, 4)


In [18]:
def build_wide_deep_model(n_users, n_items, n_features, embedding_size=64):
    """Wide and Deep recommender.

    Deep path: user + item embeddings → MLP with BatchNorm and Dropout.
    Wide path: side features (engagement, exploration, diversity, frequency) → small dense layer.
    Both paths are concatenated before the final linear output.
    """
    # --- Deep path ---
    user_input = Input(shape=(1,), name='user_input')
    user_vec   = Flatten()(Embedding(n_users, embedding_size, name='user_emb')(user_input))

    item_input = Input(shape=(1,), name='item_input')
    item_vec   = Flatten()(Embedding(n_items, embedding_size, name='item_emb')(item_input))

    deep = Concatenate()([user_vec, item_vec])
    deep = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-6))(deep)
    deep = BatchNormalization()(deep)
    deep = Dropout(0.3)(deep)
    deep = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-6))(deep)
    deep = BatchNormalization()(deep)
    deep = Dropout(0.2)(deep)

    # --- Wide path ---
    feature_input = Input(shape=(n_features,), name='feature_input')
    # Single dense layer — linear memorisation of explicit side features
    wide = Dense(32, activation='relu')(feature_input)

    # --- Combine and output ---
    combined = Concatenate()([deep, wide])
    combined = Dense(32, activation='relu')(combined)
    output   = Dense(1, activation='linear', name='output')(combined)

    return Model(inputs=[user_input, item_input, feature_input], outputs=output)


wd_model = build_wide_deep_model(n_users, n_items, n_features=X_train_features_scaled.shape[1])
wd_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
wd_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 user_input (InputLayer)     [(None, 1)]                  0         []                            
                                                                                                  
 item_input (InputLayer)     [(None, 1)]                  0         []                            
                                                                                                  
 user_emb (Embedding)        (None, 1, 64)                5551360   ['user_input[0][0]']          
                                                                                                  
 item_emb (Embedding)        (None, 1, 64)                320       ['item_input[0][0]']          
                                                                                            

In [19]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

print("Training Wide and Deep model...")
wd_history = wd_model.fit(
    [X_train_ncf[:, 0], X_train_ncf[:, 1], X_train_features_scaled], y_train_ncf,
    validation_data=(
        [X_test_ncf[:, 0], X_test_ncf[:, 1], X_test_features_scaled], y_test_ncf
    ),
    epochs=50,
    batch_size=512,
    callbacks=[early_stop, reduce_lr],
    verbose=1,
)
print(f"Wide and Deep training complete — {len(wd_history.history['loss'])} epochs.")

Training Wide and Deep model...
Epoch 1/50
286/286 [==============================] - 27s 86ms/step - loss: 7.1047 - mae: 1.9917 - val_loss: 7.0751 - val_mae: 2.0296 - lr: 0.0010
Epoch 2/50
286/286 [==============================] - 24s 84ms/step - loss: 4.7862 - mae: 1.6580 - val_loss: 5.9076 - val_mae: 1.8631 - lr: 0.0010
Epoch 3/50
286/286 [==============================] - 23s 80ms/step - loss: 2.4825 - mae: 1.1585 - val_loss: 6.3389 - val_mae: 1.9274 - lr: 0.0010
Epoch 4/50
286/286 [==============================] - 24s 85ms/step - loss: 1.4467 - mae: 0.9093 - val_loss: 6.1890 - val_mae: 1.9091 - lr: 0.0010
Epoch 5/50
286/286 [==============================] - 24s 85ms/step - loss: 0.9582 - mae: 0.7334 - val_loss: 5.9923 - val_mae: 1.8763 - lr: 0.0010
Epoch 6/50
286/286 [==============================] - 23s 82ms/step - loss: 0.7097 - mae: 0.6316 - val_loss: 5.8729 - val_mae: 1.8613 - lr: 5.0000e-04
Epoch 7/50
286/286 [==============================] - 24s 84ms/step - loss: 0.4916

In [20]:
# Predict with all three inputs (user, item, features)
y_pred_wd = wd_model.predict(
    [X_test_ncf[:, 0], X_test_ncf[:, 1], X_test_features_scaled], verbose=0
).flatten()

rmse_wd = np.sqrt(mean_squared_error(y_test_ncf, y_pred_wd))
mae_wd  = mean_absolute_error(y_test_ncf, y_pred_wd)

rmse_improvement_wd = (rmse_user - rmse_wd) / rmse_user * 100
mae_improvement_wd  = (mae_user  - mae_wd)  / mae_user  * 100

print(f"Wide & Deep — RMSE={rmse_wd:.4f} ({rmse_improvement_wd:+.2f}% vs baseline)  "
      f"MAE={mae_wd:.4f} ({mae_improvement_wd:+.2f}% vs baseline)")

wd_metrics = {
    'rmse': float(rmse_wd),
    'mae':  float(mae_wd),
    'rmse_improvement_pct': float(rmse_improvement_wd),
    'mae_improvement_pct':  float(mae_improvement_wd),
}

Wide & Deep — RMSE=2.4234 (+27.94% vs baseline)  MAE=1.8613 (+26.00% vs baseline)


In [21]:
# Save Wide & Deep model as .h5 — required format for the FastAPI backend
wd_model_path = '../artifacts/models/recommendation_wide_deep.h5'
wd_model.save(wd_model_path)

# Save the feature scaler so inference can apply the same transformation
scaler_path = '../artifacts/scalers/recommendation_feature_scaler.pkl'
joblib.dump(feature_scaler, scaler_path)

# Save the label encoders so inference can convert raw IDs to integer indices
user_enc_path = '../artifacts/encoders/recommendation_user_encoder.pkl'
item_enc_path = '../artifacts/encoders/recommendation_item_encoder.pkl'
joblib.dump(user_encoder, user_enc_path)
joblib.dump(item_encoder, item_enc_path)

wd_metadata = {
    'model_version':      MODEL_VERSION,
    'model_type':         'Wide and Deep Recommender',
    'created_at':         datetime.now().isoformat(),
    'n_users':            int(n_users),
    'n_items':            int(n_items),
    'embedding_size':     64,
    'feature_columns':    feature_cols,
    'metrics':            wd_metrics,
    'training_epochs':    len(wd_history.history['loss']),
    'artifacts': {
        'model':        wd_model_path,
        'scaler':       scaler_path,
        'user_encoder': user_enc_path,
        'item_encoder': item_enc_path,
    },
}
with open('../artifacts/models/recommendation_wide_deep_metadata.json', 'w') as f:
    json.dump(wd_metadata, f, indent=2)

print(f"Wide & Deep model saved : {wd_model_path}")
print(f"Feature scaler saved    : {scaler_path}")
print(f"User encoder saved      : {user_enc_path}")
print(f"Item encoder saved      : {item_enc_path}")

Wide & Deep model saved : ../artifacts/models/recommendation_wide_deep.h5
Feature scaler saved    : ../artifacts/scalers/recommendation_feature_scaler.pkl
User encoder saved      : ../artifacts/encoders/recommendation_user_encoder.pkl
Item encoder saved      : ../artifacts/encoders/recommendation_item_encoder.pkl


## Step 8: Model Comparison and Visualisation

I compare all three models and both baselines on RMSE and MAE. The bar charts make it easy to see whether the neural models improved on the SVD baseline and by how much relative to the naive per-user mean.

In [22]:
# Build a comparison table covering baselines and all three models
comparison_df = pd.DataFrame({
    'Model': ['Baseline (Global Mean)', 'Baseline (User Mean)', 'SVD', 'NCF', 'Wide & Deep'],
    'RMSE':  [rmse_global, rmse_user, rmse_svd, rmse_ncf, rmse_wd],
    'MAE':   [mae_global,  mae_user,  mae_svd,  mae_ncf,  mae_wd],
})

# Improvement over per-user mean baseline (positive = better)
comparison_df['RMSE_improvement%'] = (rmse_user - comparison_df['RMSE']) / rmse_user * 100
comparison_df['MAE_improvement%']  = (mae_user  - comparison_df['MAE'])  / mae_user  * 100

print("Recommendation model comparison:")
print(comparison_df.round(4).to_string(index=False))

comparison_df.to_csv('../reports/recommendation_model_comparison.csv', index=False)
print(f"\nComparison saved to: ../reports/recommendation_model_comparison.csv")

Recommendation model comparison:
                 Model   RMSE    MAE  RMSE_improvement%  MAE_improvement%
Baseline (Global Mean) 2.5939 1.9211            22.8729           23.6194
  Baseline (User Mean) 3.3631 2.5151             0.0000            0.0000
                   SVD 2.6185 1.9506            22.1395           22.4459
                   NCF 2.5859 1.9239            23.1088           23.5076
           Wide & Deep 2.4234 1.8613            27.9430           25.9959

Comparison saved to: ../reports/recommendation_model_comparison.csv


In [23]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Only show the three ML models in the bars; baselines shown as reference lines
ml_df  = comparison_df[~comparison_df['Model'].str.contains('Baseline')]
x      = np.arange(len(ml_df))
colors = ['#3498db', '#2ecc71', '#e74c3c']

# RMSE comparison
axes[0].bar(x, ml_df['RMSE'], color=colors, alpha=0.85, edgecolor='white')
axes[0].axhline(y=rmse_user, color='red', linestyle='--', linewidth=2, label='User mean baseline')
axes[0].set_xticks(x)
axes[0].set_xticklabels(ml_df['Model'])
axes[0].set_ylabel('RMSE (lower is better)')
axes[0].set_title('RMSE Comparison', fontweight='bold', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# MAE comparison
axes[1].bar(x, ml_df['MAE'], color=colors, alpha=0.85, edgecolor='white')
axes[1].axhline(y=mae_user, color='red', linestyle='--', linewidth=2, label='User mean baseline')
axes[1].set_xticks(x)
axes[1].set_xticklabels(ml_df['Model'])
axes[1].set_ylabel('MAE (lower is better)')
axes[1].set_title('MAE Comparison', fontweight='bold', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../reports/recommendation_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Comparison chart saved to: ../reports/recommendation_model_comparison.png")

Comparison chart saved to: ../reports/recommendation_model_comparison.png


## Summary

**Models trained:** SVD, Neural Collaborative Filtering, Wide and Deep

**Data used:** `transaction_level_enriched.csv` — engagement-weighted ratings with four behavioural side features.

**Key design decisions:**
- Stratified user-level split prevents cold-start users from appearing in the test set
- Engagement-weighted ratings give higher-engagement customers more influence over the model
- NCF and Wide & Deep saved as `.h5` for the FastAPI backend

**Artefacts saved:**
```
artifacts/models/
  recommendation_svd.pkl
  recommendation_ncf.h5
  recommendation_wide_deep.h5
  recommendation_*_metadata.json
artifacts/scalers/
  recommendation_feature_scaler.pkl
artifacts/encoders/
  recommendation_user_encoder.pkl
  recommendation_item_encoder.pkl
reports/
  recommendation_model_comparison.csv
  recommendation_model_comparison.png
```